In [1]:
import os
import cv2
import glob
import random
import numpy as np
from tqdm import tqdm
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Conv2DTranspose, 
                                     concatenate, BatchNormalization, Activation, 
                                     add, GlobalAveragePooling2D, Reshape, Dense, 
                                     Multiply, MultiHeadAttention, LayerNormalization)
from sklearn.model_selection import train_test_split
from IPython.display import Markdown, display

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

# Thiết lập Seed cho khâu xáo trộn dữ liệu ban đầu
SEED_VALUE = 24
random.seed(SEED_VALUE)
np.random.seed(SEED_VALUE)
tf.random.set_seed(SEED_VALUE)

# BỎ CHẾ ĐỘ TUẦN TỰ ĐỂ GIẢI PHÓNG TỐC ĐỘ GPU CHẠY SONG SONG TỐI ĐA (TĂNG TỐC ĐỘ CHO 9 CẤU HÌNH)
if 'TF_DETERMINISTIC_OPS' in os.environ:
    del os.environ['TF_DETERMINISTIC_OPS']

print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print("⚡ Đã mở khóa luồng GPU Async! Tốc độ huấn luyện ảnh nội soi sẽ đạt mức tối đa.")

IMG_HEIGHT, IMG_WIDTH, N_CHANNELS = 192, 256, 3       
BATCH_SIZE, EPOCHS, ALPHA, LEARNING_RATE = 8, 150, 1.67, 1e-3

2026-06-22 08:46:56.403231: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782118016.589706      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782118016.646265      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782118017.071177      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782118017.071220      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782118017.071223      24 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
Num GPUs Available: 1
⚡ Đã mở khóa luồng GPU Async! Tốc độ huấn luyện ảnh nội soi sẽ đạt mức tối đa.


In [2]:
# Đường dẫn tuyệt đối chính xác đến tập CVC-ClinicDB của bạn trên Kaggle
base_input_path = "/kaggle/input/datasets/quanhh42/multiresunet-datasets/CVC-ClinicDB/CVC-ClinicDB"

def load_cvc_clinicdb_data_full():
    print("\nĐang quét cấu trúc thư mục CVC-ClinicDB để bắt cặp dữ liệu...")
    all_img_dirs = glob.glob(os.path.join(base_input_path, '**', '*Original*'), recursive=True)
    all_msk_dirs = glob.glob(os.path.join(base_input_path, '**', '*Ground Truth*'), recursive=True)

    all_img_dirs = [d for d in all_img_dirs if os.path.isdir(d)]
    all_msk_dirs = [d for d in all_msk_dirs if os.path.isdir(d)]

    image_dict = {}
    for img_dir in all_img_dirs:
        for img_path in glob.glob(os.path.join(img_dir, '*.*')):
            file_stem = os.path.splitext(os.path.basename(img_path))[0] 
            if file_stem not in image_dict:
                image_dict[file_stem] = img_path

    mask_dict = {}
    for msk_dir in all_msk_dirs:
        for msk_path in glob.glob(os.path.join(msk_dir, '*.*')):
            file_stem = os.path.splitext(os.path.basename(msk_path))[0]
            if file_stem not in mask_dict:
                mask_dict[file_stem] = msk_path

    common_stems = sorted(list(set(image_dict.keys()).intersection(set(mask_dict.keys()))))
    
    if len(common_stems) == 0:
        raise ValueError("Không tìm thấy ảnh nào. Vui lòng kiểm tra lại đường dẫn base_input_path!")
        
    print(f"Đã gom và lọc trùng thành công! Tìm thấy chính xác {len(common_stems)} cặp ảnh thực tế.")
    
    X, Y = [], []
    for stem in tqdm(common_stems, desc="Loading & Processing Images"):
        img_path = image_dict[stem]
        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) 
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT)) / 255.0
        X.append(img)
        
        msk_path = mask_dict[stem]
        msk = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
        msk = cv2.resize(msk, (IMG_WIDTH, IMG_HEIGHT)) / 255.0
        Y.append(np.round(msk, 0))
        
    return np.array(X, dtype=np.float32), np.expand_dims(np.array(Y, dtype=np.float32), -1)

try:
    X, Y = load_cvc_clinicdb_data_full()
    print(f"Mảng thực nghiệm sẵn sàng! X shape: {X.shape}, Y shape: {Y.shape}")
except Exception as e:
    print(f"Thông báo tải file: {e} -> Kích hoạt mảng giả lập để bảo vệ runtime.")
    X = np.random.rand(100, IMG_HEIGHT, IMG_WIDTH, 3).astype(np.float32)
    Y = np.random.randint(0, 2, (100, IMG_HEIGHT, IMG_WIDTH, 1)).astype(np.float32)

# Chia tập dữ liệu 80% để Train và 20% để Validation nhằm đối chứng song song
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=SEED_VALUE)
print(f"✔️ Train-Test Split hoàn tất: Train = {X_train.shape[0]} ảnh | Val = {X_val.shape[0]} ảnh")


Đang quét cấu trúc thư mục CVC-ClinicDB để bắt cặp dữ liệu...
Đã gom và lọc trùng thành công! Tìm thấy chính xác 612 cặp ảnh thực tế.


Loading & Processing Images: 100%|██████████| 612/612 [00:15<00:00, 39.28it/s]


Mảng thực nghiệm sẵn sàng! X shape: (612, 192, 256, 3), Y shape: (612, 192, 256, 1)
✔️ Train-Test Split hoàn tất: Train = 489 ảnh | Val = 123 ảnh


In [3]:
def jacard(y_true, y_pred):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    return K.sum(y_true_f * y_pred_f) / (K.sum(y_true_f + y_pred_f - y_true_f * y_pred_f) + K.epsilon())

def dice_coef(y_true, y_pred):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    return (2.0 * K.sum(y_true_f * y_pred_f) + K.epsilon()) / (K.sum(y_true_f) + K.sum(y_pred_f) + K.epsilon())

def ea_ftl_loss(y_true, y_pred):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    tp = K.sum(y_true_f * y_pred_f)
    fp = K.sum((1.0 - y_true_f) * y_pred_f)
    fn = K.sum(y_true_f * (1.0 - y_pred_f))
    tversky = (tp + K.epsilon()) / (tp + 0.3 * fp + 0.7 * fn + K.epsilon())
    ftl = K.pow((1.0 - tversky), 4./3.)
    y_true_edges = tf.image.sobel_edges(y_true)
    y_pred_edges = tf.image.sobel_edges(y_pred)
    edge_loss = K.mean(K.abs(y_true_edges - y_pred_edges))
    return ftl + 0.5 * edge_loss

def conv2d_bn(x, filters, num_row, num_col, padding='same', strides=(1, 1), activation='relu'):
    x = Conv2D(filters, (num_row, num_col), strides=strides, padding=padding, use_bias=False)(x)
    x = BatchNormalization(axis=3, scale=False)(x)
    return Activation(activation)(x) if activation is not None else x

In [4]:
def MultiResBlock_factory(U, inp, use_se=False): 
    W = ALPHA * U
    short = conv2d_bn(inp, int(W*0.167)+int(W*0.333)+int(W*0.5), 1, 1, activation=None)
    c3 = conv2d_bn(inp, int(W*0.167), 3, 3); c5 = conv2d_bn(c3, int(W*0.333), 3, 3); c7 = conv2d_bn(c5, int(W*0.5), 3, 3)
    out = Activation('relu')(add([short, concatenate([c3, c5, c7], 3)]))
    if use_se:
        c_se = K.int_shape(out)[3]
        se = GlobalAveragePooling2D()(out)
        se = Reshape((1, 1, c_se))(se)
        se = Dense(c_se // 8, activation='relu', use_bias=False)(se)
        se = Dense(c_se, activation='sigmoid', use_bias=False)(se)
        out = Multiply()([out, se])
    return out

def ResPath_factory(f, length, inp, use_se=False, use_att_respath=False): 
    out = conv2d_bn(inp, f, 3, 3)
    out = add([conv2d_bn(inp, f, 1, 1, activation=None), out])
    out = Activation('relu')(BatchNormalization(axis=3)(out))
    for _ in range(length - 1):
        short = conv2d_bn(out, f, 1, 1, activation=None)
        out = Activation('relu')(add([short, conv2d_bn(out, f, 3, 3)]))
        out = BatchNormalization(axis=3)(out)
        
    # TÍCH HỢP KHỐI ATTENTION-AWARE RESPATH THEO YÊU CẦU PHẢN BIỆN CHÍ CHÓT
    if use_att_respath:
        c_att = K.int_shape(out)[3]
        gating = Conv2D(c_att, (1, 1), padding='same', activation='sigmoid')(out)
        out = Multiply()([out, gating])
        
    if use_se and not use_att_respath: 
        c_se = K.int_shape(out)[3]
        se = GlobalAveragePooling2D()(out)
        se = Reshape((1, 1, c_se))(se)
        se = Dense(c_se // 8, activation='relu', use_bias=False)(se)
        se = Dense(c_se, activation='sigmoid', use_bias=False)(se)
        out = Multiply()([out, se])
    return out

def TransformerBlock_factory(inputs):
    shape = K.int_shape(inputs); h, w, c = shape[1], shape[2], shape[3]
    x = Reshape((h * w, c))(inputs)
    attn_out = MultiHeadAttention(num_heads=4, key_dim=128)(x, x)
    x = LayerNormalization(epsilon=1e-6)(add([x, attn_out]))
    ffn_out = Dense(c)(Dense(512, activation='relu')(x))
    return Reshape((h, w, c))(LayerNormalization(epsilon=1e-6)(add([x, ffn_out])))

os.makedirs('checkpoint_ablation_clinic', exist_ok=True)
print("✔️ Khởi tạo tài nguyên nền tảng CVC-ClinicDB thành công.")

✔️ Khởi tạo tài nguyên nền tảng CVC-ClinicDB thành công.


In [5]:
# Cấu trúc hàm sinh mô hình linh hoạt động phục vụ Ablation Matrix
def build_ablation_network(use_se, use_transformer, use_att_respath):
    inputs = Input((IMG_HEIGHT, IMG_WIDTH, 3))
    
    # Encoder
    m1 = MultiResBlock_factory(32, inputs, use_se); p1 = MaxPooling2D((2,2))(m1); r1 = ResPath_factory(32, 4, m1, use_se, use_att_respath)
    m2 = MultiResBlock_factory(64, p1, use_se); p2 = MaxPooling2D((2,2))(m2); r2 = ResPath_factory(64, 3, m2, use_se, use_att_respath)
    m3 = MultiResBlock_factory(128, p2); p3 = MaxPooling2D((2,2))(m3); r3 = ResPath_factory(128, 2, m3, use_se, use_att_respath)
    m4 = MultiResBlock_factory(256, p3); p4 = MaxPooling2D((2,2))(m4); r4 = ResPath_factory(256, 1, m4, use_se, use_att_respath)
    
    # Bottleneck
    m5 = MultiResBlock_factory(512, p4)
    if use_transformer:
        m5 = TransformerBlock_factory(m5)
        
    # Decoder
    u6 = concatenate([Conv2DTranspose(256, (2,2), strides=(2,2), padding='same')(m5), r4], 3); m6 = MultiResBlock_factory(256, u6)
    u7 = concatenate([Conv2DTranspose(128, (2,2), strides=(2,2), padding='same')(m6), r3], 3); m7 = MultiResBlock_factory(128, u7)
    u8 = concatenate([Conv2DTranspose(64, (2,2), strides=(2,2), padding='same')(m7), r2], 3); m8 = MultiResBlock_factory(64, u8, use_se)
    u9 = concatenate([Conv2DTranspose(32, (2,2), strides=(2,2), padding='same')(m8), r1], 3); m9 = MultiResBlock_factory(32, u9, use_se)
    
    return Model(inputs, Conv2D(1, (1,1), activation='sigmoid')(m9))

# Bản đồ 9 cấu hình bóc tách thành phần rạch ròi, bao gồm cả Att-ResPath độc lập
ablation_tasks = [
    {"name": "Baseline MultiResUNet",                  "se": False, "trans": False, "att_res": False, "loss_type": "bce"},
    {"name": "Baseline + Transformer",                 "se": False, "trans": True,  "att_res": False, "loss_type": "bce"},
    {"name": "Baseline + SE",                          "se": True,  "trans": False, "att_res": False, "loss_type": "bce"},
    {"name": "Baseline + Att-ResPath",                 "se": False, "trans": False, "att_res": True,  "loss_type": "bce"},
    {"name": "Baseline + EA-FTL Loss",                 "se": False, "trans": False, "att_res": False, "loss_type": "ftl"},
    {"name": "Baseline + SE + Transformer",            "se": True,  "trans": True,  "att_res": False, "loss_type": "bce"},
    {"name": "Baseline + SE + EA-FTL Loss",            "se": True,  "trans": False, "att_res": False, "loss_type": "ftl"},
    {"name": "Baseline + Transformer + EA-FTL Loss",    "se": False, "trans": True,  "att_res": False, "loss_type": "ftl"},
    {"name": "Full Proposed Model (HTS-MultiResUNet)", "se": True,  "trans": True,  "att_res": True,  "loss_type": "ftl"}
]

ablation_results = []

for idx, task in enumerate(ablation_tasks, 1):
    print(f"  ĐANG CHẠY CẤU HÌNH {idx}/{len(ablation_tasks)}: {task['name']} (Full {EPOCHS} Epochs)")
    
    K.clear_session()
    model = build_ablation_network(use_se=task["se"], use_transformer=task["trans"], use_att_respath=task["att_res"])
    
    current_loss = 'binary_crossentropy' if task["loss_type"] == "bce" else ea_ftl_loss
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss=current_loss, metrics=[jacard, dice_coef])
    
    chkpt_path = f"checkpoint_ablation_clinic/task_{idx}.weights.h5"
    chkpt = tf.keras.callbacks.ModelCheckpoint(chkpt_path, monitor='val_jacard', mode='max', save_best_only=True, save_weights_only=True, verbose=0)
    
    # Huấn luyện liên tục không dừng sớm, giải phóng tối đa tài nguyên Async GPU
    history = model.fit(X_train, Y_train, validation_data=(X_val, Y_val), batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=[chkpt], verbose=1)
    
    best_j = max(history.history['val_jacard']) * 100
    best_d = max(history.history['val_dice_coef']) * 100
    
    ablation_results.append({
        "name": task["name"],
        "se": "✓" if task["se"] else "✗",
        "trans": "✓" if task["trans"] else "✗",
        "att_res": "✓" if task["att_res"] else "✗",
        "loss": "EA-FTL" if task["loss_type"] == "ftl" else "BCE",
        "jaccard": f"{best_j:.2f}",
        "dice": f"{best_d:.2f}"
    })
    print(f"✔️ Xong nhánh {idx}! Đạt Jaccard = {best_j:.2f}% | Dice = {best_d:.2f}%")

# --- TỰ ĐỘNG XUẤT BẢNG SO SÁNH ĐỊNH LƯỢNG MARKDOWN HOÀN CHỈNH ---
print("\n TIẾN TRÌNH KẾT THÚC! BẢNG SỐ LIỆU NGHIÊN CỨU THÀNH PHẦN ABLATION STUDY (CVC-ClinicDB):")

markdown_table = """| Configuration | SE-Block | Transformer | Att-ResPath | EA-FTL Loss | Jaccard (%) ↑ | Dice (%) ↑ |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
"""
for res in ablation_results:
    markdown_table += f"| {res['name']} | {res['se']} | {res['trans']} | {res['att_res']} | {res['loss']} | {res['jaccard']} | {res['dice']} |\n"

display(Markdown(markdown_table))

  ĐANG CHẠY CẤU HÌNH 1/9: Baseline MultiResUNet (Full 150 Epochs)


I0000 00:00:1782118069.849058      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/150


I0000 00:00:1782118098.971313      70 service.cc:152] XLA service 0x7ccdf00041b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782118098.971351      70 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1782118103.573777      70 cuda_dnn.cc:529] Loaded cuDNN version 91002


 1/62 ━━━━━━━━━━━━━━━━━━━━ 1:10:27 69s/step - dice_coef: 0.1282 - jacard: 0.0685 - loss: 0.4350

I0000 00:00:1782118142.112323      70 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


62/62 ━━━━━━━━━━━━━━━━━━━━ 118s 796ms/step - dice_coef: 0.2009 - jacard: 0.1127 - loss: 0.2882 - val_dice_coef: 0.1347 - val_jacard: 0.0726 - val_loss: 0.8875
Epoch 2/150
62/62 ━━━━━━━━━━━━━━━━━━━━ 10s 163ms/step - dice_coef: 0.2923 - jacard: 0.1727 - loss: 0.2228 - val_dice_coef: 0.1461 - val_jacard: 0.0796 - val_loss: 7.4253
Epoch 3/150
62/62 ━━━━━━━━━━━━━━━━━━━━ 10s 163ms/step - dice_coef: 0.3491 - jacard: 0.2129 - loss: 0.2041 - val_dice_coef: 0.1567 - val_jacard: 0.0858 - val_loss: 8.4755
Epoch 4/150
62/62 ━━━━━━━━━━━━━━━━━━━━ 10s 163ms/step - dice_coef: 0.3976 - jacard: 0.2495 - loss: 0.1917 - val_dice_coef: 0.1698 - val_jacard: 0.0938 - val_loss: 4.2446
Epoch 5/150
62/62 ━━━━━━━━━━━━━━━━━━━━ 10s 162ms/step - dice_coef: 0.4217 - jacard: 0.2692 - loss: 0.1832 - val_dice_coef: 0.2139 - val_jacard: 0.1216 - val_loss: 1.0533
Epoch 6/150
62/62 ━━━━━━━━━━━━━━━━━━━━ 10s 163ms/step - dice_coef: 0.4914 - jacard: 0.3272 - loss: 0.1625 - val_dice_coef: 0.2280 - val_jacard: 0.1304 - val_loss

| Configuration | SE-Block | Transformer | Att-ResPath | EA-FTL Loss | Jaccard (%) ↑ | Dice (%) ↑ |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| Baseline MultiResUNet | ✗ | ✗ | ✗ | BCE | 73.14 | 83.71 |
| Baseline + Transformer | ✗ | ✓ | ✗ | BCE | 75.99 | 85.85 |
| Baseline + SE | ✓ | ✗ | ✗ | BCE | 75.40 | 85.41 |
| Baseline + Att-ResPath | ✗ | ✗ | ✓ | BCE | 71.35 | 82.66 |
| Baseline + EA-FTL Loss | ✗ | ✗ | ✗ | EA-FTL | 73.07 | 83.97 |
| Baseline + SE + Transformer | ✓ | ✓ | ✗ | BCE | 78.74 | 87.84 |
| Baseline + SE + EA-FTL Loss | ✓ | ✗ | ✗ | EA-FTL | 75.09 | 85.14 |
| Baseline + Transformer + EA-FTL Loss | ✗ | ✓ | ✗ | EA-FTL | 75.56 | 85.80 |
| Full Proposed Model (HTS-MultiResUNet) | ✓ | ✓ | ✓ | EA-FTL | 75.60 | 85.31 |
